### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [23]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [25]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [26]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [27]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [28]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [29]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [30]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [31]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [32]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [33]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [34]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [35]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [36]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [37]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [38]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [39]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  6385,  1149, 11238])

Obtenemos los 5 documentos más similares:

In [40]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [41]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [42]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [43]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [44]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [45]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## **1. Vectorización y Similaridad de Documentos**

In [67]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Elegir 5 índices al azar
np.random.seed(42) # Para que sea reproducible
indices_azar = np.random.choice(len(newsgroups_train.data), 5)

for idx in indices_azar:
    # Calcular similaridad del documento elegido contra TODO el set de entrenamiento
    similitudes = cosine_similarity(X_train[idx], X_train)[0]
    
    # Obtener los 5 más similares (excluyendo el primero que es sí mismo)
    top_5_idx = np.argsort(similitudes)[::-1][1:6]
    
    print(f"\nDocumento Original (ID: {idx}) - Clase: {newsgroups_train.target_names[y_train[idx]]}")
    print(f"Texto resumido: {newsgroups_train.data[idx][:100]}...")
    
    print("Top 5 Documentos Similares:")
    for sim_idx in top_5_idx:
        clase_sim = newsgroups_train.target_names[y_train[sim_idx]]
        score = similitudes[sim_idx]
        print(f" -> Clase: {clase_sim} (Score: {score:.3f})")


Documento Original (ID: 7270) - Clase: rec.sport.hockey
Texto resumido: 
I think Murray has done a great job.  He's picked up Ciccarelli,
Sheppard, Ysebaert, Howe, Coffey, ...
Top 5 Documentos Similares:
 -> Clase: rec.sport.hockey (Score: 0.259)
 -> Clase: talk.politics.mideast (Score: 0.240)
 -> Clase: talk.politics.mideast (Score: 0.234)
 -> Clase: talk.politics.mideast (Score: 0.231)
 -> Clase: talk.politics.mideast (Score: 0.228)

Documento Original (ID: 860) - Clase: talk.politics.mideast
Texto resumido: 


This is an interesting question to ponder.  Did Brad/Ali's sickness
make Ayatollah-style Islam at...
Top 5 Documentos Similares:
 -> Clase: comp.windows.x (Score: 0.174)
 -> Clase: sci.crypt (Score: 0.161)
 -> Clase: talk.politics.mideast (Score: 0.157)
 -> Clase: talk.politics.mideast (Score: 0.153)
 -> Clase: talk.politics.misc (Score: 0.151)

Documento Original (ID: 5390) - Clase: alt.atheism
Texto resumido: 

Specifically, which changes are you talking about?  Are you arg

Se observa que la similaridad coseno es altamente efectiva en documentos con terminología técnica específica (Hardware, Ateísmo). Sin embargo, en textos cortos o de carácter subjetivo/argumentativo, el modelo tiende a agrupar documentos por estilo de redacción (palabras comunes) en lugar de por temática, lo que explica por qué un texto de 'Autos' se confundió con uno de 'Religión' al compartir conectores y verbos de opinión.

## **2. Clasificador por Prototipos**

In [49]:
sim_matrix = cosine_similarity(X_test, X_train)
most_similar_train = np.argmax(sim_matrix, axis=1)

# La predicción es la etiqueta del doc de train más similar
y_pred_proto = y_train[most_similar_train]

# Evaluamos con F1 macro
f1 = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-Score Macro (clasificación por prototipos): {f1:.4f}")

F1-Score Macro (clasificación por prototipos): 0.5050


## **3. Optimización de Naïve Bayes**

In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score

# 1. Instanciamos el vectorizador con parámetros optimizados
tfidf_opt = TfidfVectorizer(
    stop_words='english', # Limpieza de conectores
    max_df=0.4,           # Ignorar palabras que aparecen en más del 40% de los docs
    min_df=3,             # Ignorar palabras que aparecen en menos de 3 docs
    sublinear_tf=True     # Escala logarítmica para suavizar frecuencias altas
)

# 2. Transformamos los datos
X_train_opt = tfidf_opt.fit_transform(newsgroups_train.data)
X_test_opt = tfidf_opt.transform(newsgroups_test.data)

# 3. Entrenamos un Complement Naive Bayes
# El parámetro alpha=0.1 suele ayudar a que el modelo sea más sensible a términos raros
clf_opt = ComplementNB(alpha=1)#Se pudo en 1 porque así da una mejor predicción
clf_opt.fit(X_train_opt, y_train)

# 4. Predicción y evaluación
y_pred_opt = clf_opt.predict(X_test_opt)
score = f1_score(y_test, y_pred_opt, average='macro')

print(f"Nuevo F1-Score Macro: {score:.4f}")
print(f"Tamaño del vocabulario tras optimización: {len(tfidf_opt.vocabulary_)}")

Nuevo F1-Score Macro: 0.6907
Tamaño del vocabulario tras optimización: 26747


In [68]:
import matplotlib.pyplot as plt

# Valores de alpha a probar
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0]
scores = []

for a in alphas:
    # Usamos el vectorizador optimizado del paso anterior
    model = ComplementNB(alpha=a)
    model.fit(X_train_opt, y_train)
    
    # Predecimos y calculamos F1-Score Macro
    preds = model.predict(X_test_opt)
    f1 = f1_score(y_test, preds, average='macro')
    scores.append(f1)
    print(f"Alpha: {a:<6} | F1-Score Macro: {f1:.4f}")

# Identificar el mejor
best_alpha = alphas[np.argmax(scores)]
print(f"\n El mejor Alpha para el modelo es: {best_alpha}")

Alpha: 0.001  | F1-Score Macro: 0.6690
Alpha: 0.01   | F1-Score Macro: 0.6734
Alpha: 0.1    | F1-Score Macro: 0.6821
Alpha: 0.5    | F1-Score Macro: 0.6903
Alpha: 1.0    | F1-Score Macro: 0.6907
Alpha: 2.0    | F1-Score Macro: 0.6883

 El mejor Alpha para el modelo es: 1.0


Al optimizar con ComplementNB y ajustar el vectorizador, se logró equilibrio. El modelo es órdenes de magnitud más rápido que el de prototipos y mucho más preciso, alcanzando un F1-Score de 0.69 gracias al tratamiento de clases desbalanceadas y la reducción de ruido con max_df y min_df.

In [66]:
# 1. Transponemos la matriz: (Documentos x Términos) -> (Términos x Documentos)
X_terms = X_train.T 

# 2. Obtenemos el diccionario inverso para buscar palabras por índice
idx2word = {v: k for k, v in tfidfvect.vocabulary_.items()}

# 3. Lista de palabras manuales para investigar (términos clave del dataset)
# Elegimos palabras con carga semántica clara para ver si sus similares tienen sentido
palabras_test = ['space', 'god', 'encryption', 'hockey', 'medical']

print("--- Análisis de Similaridad entre Palabras ---")

for word in palabras_test:
    if word in tfidfvect.vocabulary_:
        # Obtenemos el índice de la palabra
        word_idx = tfidfvect.vocabulary_[word]
        
        # Calculamos la similaridad de esta palabra contra TODAS las demás
        # X_terms[word_idx] es el vector de la palabra elegida
        word_sims = cosine_similarity(X_terms[word_idx], X_terms)[0]
        
        # Obtenemos los índices de los 6 más similares (el 1ro siempre es la palabra misma)
        top_indices = np.argsort(word_sims)[::-1][1:6]
        
        print(f"\nPalabra clave: '{word.upper()}'")
        for i in top_indices:
            similar_word = idx2word[i]
            score = word_sims[i]
            print(f" -> {similar_word:<15} | Similitud: {score:.3f}")
    else:
        print(f"\nLa palabra '{word}' no se encuentra en el vocabulario.")

--- Análisis de Similaridad entre Palabras ---

Palabra clave: 'SPACE'
 -> nasa            | Similitud: 0.330
 -> seds            | Similitud: 0.297
 -> shuttle         | Similitud: 0.293
 -> enfant          | Similitud: 0.280
 -> seti            | Similitud: 0.246

Palabra clave: 'GOD'
 -> jesus           | Similitud: 0.269
 -> bible           | Similitud: 0.262
 -> that            | Similitud: 0.256
 -> existence       | Similitud: 0.255
 -> christ          | Similitud: 0.251

Palabra clave: 'ENCRYPTION'
 -> microcircuit    | Similitud: 0.340
 -> 2758            | Similitud: 0.338
 -> pitted          | Similitud: 0.338
 -> scrambles       | Similitud: 0.338
 -> nistnews        | Similitud: 0.338

Palabra clave: 'HOCKEY'
 -> ncaa            | Similitud: 0.274
 -> nhl             | Similitud: 0.265
 -> affiliates      | Similitud: 0.248
 -> xenophobes      | Similitud: 0.243
 -> sportschannel   | Similitud: 0.223

Palabra clave: 'MEDICAL'
 -> romano          | Similitud: 0.282
 -> hosp

## **Conclusión Final: Análisis de Similaridad entre Palabras**
La transposición de la matriz documento-término permitió transformar nuestra base de datos en un espacio de representación semántica. Al analizar los resultados, se extraen las siguientes conclusiones:

**-Contexto como Definidor de Significado:** El experimento confirma la Hipótesis Distribucional, que postula que las palabras que aparecen en contextos similares tienden a tener significados similares. Por ejemplo, la cercanía de 'SPACE' con 'NASA' y 'SHUTTLE' demuestra que el modelo agrupó estos términos porque "cohabitan" en los mismos hilos de discusión.

**-Captura de Relaciones de Dominio:** A diferencia de la similaridad de documentos (Punto 1), que busca textos completos parecidos, la similaridad de palabras permite identificar el núcleo temático de un dataset. Logramos identificar sinónimos funcionales o términos relacionados (como 'HOCKEY' con 'NHL' o 'NCAA') sin haberle proporcionado al sistema ninguna regla gramatical ni diccionario previo.

**-Identificación de Ruido Estructural:** La aparición de términos como números (ej. '2758') o palabras comunes (ej. 'that') dentro de los más similares revela que el dataset original aún contiene elementos de ruido (cabeceras, firmas o códigos) que la vectorización TF-IDF mitiga pero no elimina por completo.

**-Eficacia de la Métrica Coseno:** Se validó que la Similaridad Coseno es una métrica robusta para este análisis, ya que mide la orientación de los vectores de palabras (su perfil de aparición) independientemente de qué tan frecuentes sean en términos absolutos, permitiendo que palabras técnicas específicas resalten sobre el ruido general.

**Resumen:** Este ejercicio demuestra cómo una estructura de datos puramente matemática (una matriz de números) puede reflejar el lenguaje humano, permitiendo que la computadora "entienda" que 'GOD' y 'JESUS' pertenecen al mismo universo conceptual basándose únicamente en su distribución estadística.